# Model Training — NIH ChestX-Ray14 Multi-Label Classification

**Project:** Multi-Label Thoracic Disease Classification using Deep Learning  
**Dataset:** NIH ChestX-Ray14 — 112,120 frontal chest X-ray images, 14 disease classes  

---

## Notebook Overview

This notebook trains and evaluates five deep learning architectures for multi-label chest disease classification:

| Model | Type | Input Size |
|-------|------|------------|
| ResNet-50 | CNN | 224×224 |
| VGG-19 | CNN | 224×224 |
| Swin Transformer V2-S | Transformer | 256×256 |
| Vision Transformer (ViT-B/16) | Transformer | 224×224 |
| Inception V3 | CNN | 299×299 |

**Key design decisions:**
- `BCEWithLogitsLoss` with `pos_weight` to handle severe class imbalance
- `AdamW` optimizer with `CosineAnnealingLR` scheduler
- Mixed precision training (`autocast` + `GradScaler`) for faster GPU utilization
- Per-class optimal threshold selection on validation set before test evaluation
- Early stopping based on validation AUC to prevent overfitting

> **Requires:** `data_essentials.pth` from `02_preprocessing.ipynb`

## 1. Imports & Device Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np
from torch.amp import autocast, GradScaler
from torch import device, cuda, nn, no_grad, tensor, load, save, sigmoid, cat, float32
from torch.utils.data import Dataset, DataLoader
from torchvision.utils import make_grid
from torchvision.models import (
    resnet50, vit_b_16, swin_v2_s, inception_v3, vgg19,
    ResNet50_Weights, ViT_B_16_Weights, Swin_V2_S_Weights,
    Inception_V3_Weights, VGG19_Weights
)
import torchvision.transforms as transforms
from torch.optim import AdamW, lr_scheduler
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score,
    recall_score, multilabel_confusion_matrix
)
import seaborn as sns
import time
from warnings import filterwarnings

filterwarnings("ignore")
%matplotlib inline

# Use GPU if available
dev = device("cuda" if cuda.is_available() else "cpu")
dev

## 2. Load Preprocessed Data & Configuration

All preprocessing artifacts (train/val/test splits, disease labels, pos_weights)
are loaded from the single checkpoint file produced by `02_preprocessing.ipynb`.

In [ ]:
# Load all preprocessing artifacts saved from 02_preprocessing.ipynb
checkpoint = load(r"data_essentials.pth", map_location=dev, weights_only=False)
checkpoint.keys()

In [ ]:
# Verify GPU availability and name
print(cuda.is_available())
print(cuda.get_device_name(0))

In [ ]:
# Path where best model weights will be saved after training
SAVED_MODELS_FOLDER = r"C:\Users\ibrah.HIMA\OneDrive\Desktop\Full AI\My Reserch Papers\chest-xray-multilabel-classification\saved_models"

# Collect paths for all 12 image subfolders (images_001 to images_012)
IMG_FOLDERS = checkpoint["img_folders"]

# Number of disease classes (14 diseases, excluding 'No Finding')
NUM_OF_CLASSES = checkpoint["num_of_classes"]

# Sorted list of 14 disease names — order must match label encoding
ALL_DISEASES = checkpoint["all_diseases"]

# Training hyperparameters
BATCH_SIZE = 128
EPOCHS     = 30
PATIENCE   = 5       # early stopping: stop if no improvement for 5 epochs
LR         = 1e-4    # initial learning rate for AdamW

# ImageNet normalization constants — required for all pretrained models
MEAN_NORM = [0.485, 0.456, 0.406]
STD_NORM  = [0.229, 0.224, 0.225]

# Per-class positive weights to address class imbalance
# Formula: pos_weight[i] = negative_samples[i] / positive_samples[i]
# Higher weight = rarer disease = stronger penalty for missing it
POS_WEIGHTS = checkpoint["pos_weights"]

# Train / validation / test DataFrames (patient-level split from NIH)
train_df = checkpoint["train_df"]
val_df   = checkpoint["val_df"]
test_df  = checkpoint["test_df"]

## 3. Data Augmentation & Transforms

Training transforms include mild augmentation suitable for chest X-rays:
- `RandomRotation(10)` — simulates slight patient positioning variation
- `RandomHorizontalFlip` — acceptable since lung anatomy is roughly symmetric

**Excluded intentionally:** `ColorJitter` — contrast and brightness carry diagnostic
information in X-rays and must not be altered artificially.

Validation and test transforms use only resize and normalization (no augmentation).

In [ ]:
def get_transforms(size=224, In_Train=True):
    """
    Build a transform pipeline for chest X-ray images.

    Parameters
    ----------
    size     : int — target image size (default 224; use 256 for Swin, 299 for Inception)
    In_Train : bool — if True, applies augmentation; if False, resize + normalize only

    Returns
    -------
    torchvision.transforms.Compose
    """
    if In_Train:
        return transforms.Compose([
            transforms.Resize((size, size)),
            transforms.RandomRotation(10),        # mild rotation for position variance
            transforms.RandomHorizontalFlip(),    # acceptable for bilateral lung anatomy
            transforms.ToTensor(),
            transforms.Normalize(mean=MEAN_NORM, std=STD_NORM)
        ])
    else:
        return transforms.Compose([
            transforms.Resize((size, size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=MEAN_NORM, std=STD_NORM)
        ])

## 4. Custom Dataset

A custom `Dataset` is required here because the NIH images are spread across
12 separate folders and the labels are stored in a CSV — not in folder structure.

The `_encode_labels` method converts pipe-separated disease strings like
`'Pneumonia|Effusion'` into 14-dimensional binary vectors:
`[0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0]`

In [ ]:
class ChestXRayDataset(Dataset):
    """
    Custom PyTorch Dataset for the NIH ChestX-Ray14 multi-label classification task.

    Loads images from multiple folders and encodes pipe-separated disease labels
    into binary vectors of length NUM_OF_CLASSES.
    """

    def __init__(self, df, img_folders, transforms=None):
        self.df          = df.reset_index(drop=True)
        self.img_folders = img_folders
        self.transforms  = transforms
        # Pre-encode all labels once at init to avoid repeated string parsing
        self.labels      = self._encode_labels()

    def _encode_labels(self):
        """
        Convert 'Finding Labels' column to multi-hot binary vectors.
        'No Finding' maps to all-zeros vector.
        """
        encode = []
        for label in self.df["Finding Labels"]:
            diseases_in_img = label.split("|")
            vector = [0.0] * NUM_OF_CLASSES
            for disease in diseases_in_img:
                if disease in ALL_DISEASES:
                    idx = ALL_DISEASES.index(disease)
                    vector[idx] = 1.0
            encode.append(vector)
        return encode

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        img_name = self.df.loc[index, "Image Index"]

        # Search across all 12 folders to find the image file
        img = None
        for folder in IMG_FOLDERS:
            img_path = os.path.join(folder, img_name)
            if os.path.exists(img_path):
                img = Image.open(img_path).convert("RGB")
                break

        if self.transforms:
            img = self.transforms(img)

        # Return image tensor and its 14-dimensional binary label vector
        label = tensor(self.labels[index], dtype=float32)
        return img, label


# Standard 224x224 datasets (ResNet, VGG, ViT)
train_dataset = ChestXRayDataset(train_df, IMG_FOLDERS, transforms=get_transforms())
val_dataset   = ChestXRayDataset(val_df,   IMG_FOLDERS, get_transforms(In_Train=False))
test_dataset  = ChestXRayDataset(test_df,  IMG_FOLDERS, get_transforms(In_Train=False))

# Inception V3 requires 299x299 input
inc_train_dataset = ChestXRayDataset(train_df, IMG_FOLDERS, get_transforms(299))
inc_val_dataset   = ChestXRayDataset(val_df,   IMG_FOLDERS, get_transforms(299, In_Train=False))
inc_test_dataset  = ChestXRayDataset(test_df,  IMG_FOLDERS, get_transforms(299, In_Train=False))

# Swin Transformer V2 requires 256x256 input
swin_train_dataset = ChestXRayDataset(train_df, IMG_FOLDERS, get_transforms(256))
swin_val_dataset   = ChestXRayDataset(val_df,   IMG_FOLDERS, get_transforms(256, In_Train=False))
swin_test_dataset  = ChestXRayDataset(test_df,  IMG_FOLDERS, get_transforms(256, In_Train=False))

## 5. DataLoaders

- `shuffle=True` for training to prevent the model from learning data order
- `num_workers=4` for parallel data loading
- `pin_memory=True` for faster CPU-to-GPU transfer

In [ ]:
# Standard loaders — used for ResNet, VGG, ViT
train_dl = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
val_dl   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_dl  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

# Inception V3 loaders — 299x299
inc_train_dl = DataLoader(inc_train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
inc_val_dl   = DataLoader(inc_val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
inc_test_dl  = DataLoader(inc_test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

# Swin Transformer loaders — 256x256
swin_train_dl = DataLoader(swin_train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
swin_val_dl   = DataLoader(swin_val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
swin_test_dl  = DataLoader(swin_test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

## 6. Helper Functions

In [ ]:
def denormalize(data):
    """
    Reverse ImageNet normalization for visualization.
    Handles both single images (3D) and batches (4D).
    """
    mean = tensor(MEAN_NORM).to(data.device)
    std  = tensor(STD_NORM).to(data.device)

    if data.ndimension() == 3:
        res = data * std[:, None, None] + mean[:, None, None]
    elif data.ndimension() == 4:
        res = data * std[None, :, None, None] + mean[None, :, None, None]
    else:
        return data

    return res.clamp(0, 1)

## 7. Sample Visualization

Visual inspection of a batch from each split to confirm correct loading and normalization.

In [ ]:
show_train_dl = DataLoader(train_dataset, batch_size=16, shuffle=True)
show_val_dl   = DataLoader(val_dataset,   batch_size=16, shuffle=False)
show_test_dl  = DataLoader(test_dataset,  batch_size=16, shuffle=False)

def Show_Sample(dl):
    """Display a 4-column grid of denormalized X-ray images from the given DataLoader."""
    _, ax = plt.subplots(figsize=(12, 10))
    imgs, labels = next(iter(dl))
    imgs = denormalize(imgs)
    ax.set_yticks([])
    ax.set_xticks([])
    grid = make_grid(imgs, 4).permute(1, 2, 0).numpy()
    ax.imshow(grid)

Show_Sample(show_train_dl)

In [ ]:
Show_Sample(show_val_dl)

In [ ]:
Show_Sample(show_test_dl)

## 8. Visualization & Evaluation Functions

In [ ]:
def Show_Curves(model_history):
    """
    Plot training and validation Loss and AUC curves over epochs.
    Saved to disk as 'AUC_LOSS_Curves.png'.

    Parameters
    ----------
    model_history : list of dicts with keys train_loss, val_loss, train_auc, val_auc
    """
    train_loss_hist = [x["train_loss"] for x in model_history]
    val_loss_hist   = [x["val_loss"]   for x in model_history]
    train_auc_hist  = [x["train_auc"]  for x in model_history]
    val_auc_hist    = [x["val_auc"]    for x in model_history]

    _, ax = plt.subplots(1, 2, figsize=(14, 5))

    ax[0].plot(train_loss_hist, label="Train",      linewidth=2)
    ax[0].plot(val_loss_hist,   label="Validation", linewidth=2, linestyle="--")
    ax[0].set_title("Loss Over Epochs", fontsize=13, fontweight="bold")
    ax[0].set_ylabel("Loss")
    ax[0].set_xlabel("Epoch")
    ax[0].legend()
    ax[0].grid(alpha=0.3)

    ax[1].plot(train_auc_hist, label="Train",      linewidth=2)
    ax[1].plot(val_auc_hist,   label="Validation", linewidth=2, linestyle="--")
    ax[1].set_title("AUC Over Epochs", fontsize=13, fontweight="bold")
    ax[1].set_ylabel("AUC")
    ax[1].set_xlabel("Epoch")
    ax[1].legend()
    ax[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('AUC_LOSS_Curves.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
def Show_MCM(labels, preds_binary):
    """
    Display per-disease confusion matrices using seaborn heatmaps.

    For each of the 14 diseases, shows a 2x2 confusion matrix:
    TN | FP
    FN | TP

    In medical imaging, FN (missed disease) is the most critical error.
    Saved to disk as 'diseases_confusion_matrices.png'.

    Parameters
    ----------
    labels       : numpy array, shape (N, 14) — ground truth binary labels
    preds_binary : numpy array, shape (N, 14) — binary predictions after thresholding
    """
    mcm = multilabel_confusion_matrix(labels, preds_binary)

    fig, axes = plt.subplots(3, 5, figsize=(20, 12))
    axes = axes.flatten()

    for i, (matrix, disease) in enumerate(zip(mcm, ALL_DISEASES)):
        sns.heatmap(
            matrix, annot=True, fmt='d', cmap='Blues',
            ax=axes[i],
            xticklabels=['Pred 0', 'Pred 1'],
            yticklabels=['True 0', 'True 1']
        )
        axes[i].set_title(disease, fontsize=10)

    # Remove the 15th unused subplot (3x5 grid has 15 slots for 14 diseases)
    fig.delaxes(axes[14])
    plt.suptitle('Confusion Matrix per Disease', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('diseases_confusion_matrices.png', dpi=300, bbox_inches='tight')
    plt.show()

## 9. Prediction & Threshold Optimization

After training, we run inference on the validation set and find the optimal
classification threshold for each disease independently.

**Why per-class thresholds?**  
The default threshold of 0.5 is rarely optimal for imbalanced medical data.
By searching thresholds from 0.1 to 0.9, we maximize the F1 score for each
disease individually. These thresholds are then applied to the test set.

In [ ]:
def model_predict(model, test_dl):
    """
    Run inference on a DataLoader and find optimal per-class thresholds.

    Thresholds are found by maximizing F1 score for each disease class
    separately. This is always called on the VALIDATION set first, and
    the resulting thresholds are then applied to the TEST set.

    Parameters
    ----------
    model   : trained PyTorch model in eval mode
    test_dl : DataLoader to run inference on

    Returns
    -------
    all_labels       : numpy array (N, 14) — ground truth
    all_preds        : numpy array (N, 14) — float probabilities from sigmoid
    best_thresholds  : list of 14 floats — optimal threshold per disease
    """
    all_preds  = []
    all_labels = []

    model.eval()
    with no_grad():
        for batch in test_dl:
            imgs, labels = batch
            imgs   = imgs.to(dev)
            labels = labels.to(dev)

            with autocast(device_type='cuda'):
                outputs = model(imgs)

            # Convert logits to probabilities — each value is independent (multi-label)
            probs = sigmoid(outputs).float()

            all_preds.append(probs.detach().cpu())
            all_labels.append(labels.detach().cpu())

    all_preds  = cat(all_preds).numpy()
    all_labels = cat(all_labels).numpy()

    # Find optimal threshold per disease by maximizing F1
    best_thresholds = []
    for i in range(14):
        best_t  = 0.5
        best_f1 = 0
        for t in np.arange(0.1, 0.9, 0.05):
            preds = (all_preds[:, i] > t).astype(int)
            f1    = f1_score(all_labels[:, i], preds, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_t  = t
        best_thresholds.append(best_t)

    print("Best thresholds per disease:", best_thresholds)
    return all_labels, all_preds, best_thresholds

## 10. Classifier Replacement

Each pretrained model is modified by replacing its final classification layer
with a new `Linear(in_features, 14)` layer — one output per disease.

A `Dropout(0.3)` layer is added before the linear layer in all models
to regularize and reduce overfitting.

**Note on Inception V3:** Has an auxiliary classifier (AuxLogits) used during
training for gradient flow. Both the main and auxiliary classifiers are replaced.
The auxiliary loss is weighted by 0.4 (standard Inception training practice).

In [ ]:
def replace_classifier(model, name):
    """
    Replace the final classification layer for any supported architecture.

    Supported: VGG, Inception, ResNet, VisionTransformer, SwinTransformer

    Parameters
    ----------
    model : pretrained PyTorch model
    name  : str — model class name used to identify architecture type

    Returns
    -------
    model with replaced classifier
    """
    if "VGG" in name:
        # VGG: replace the last element of the Sequential classifier
        model.classifier[6] = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(model.classifier[6].in_features, NUM_OF_CLASSES)
        )

    elif "Inception" in name:
        # Inception V3: replace both main fc and auxiliary classifier
        model.fc = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(model.fc.in_features, NUM_OF_CLASSES)
        )
        if model.AuxLogits is not None:
            model.AuxLogits.fc = nn.Sequential(
                nn.Dropout(p=0.3),
                nn.Linear(model.AuxLogits.fc.in_features, NUM_OF_CLASSES)
            )

    elif "ResNet" in name:
        model.fc = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(model.fc.in_features, NUM_OF_CLASSES)
        )

    elif "VisionTransformer" in name:
        model.heads = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(model.heads.head.in_features, NUM_OF_CLASSES)
        )

    elif "SwinTransformer" in name:
        model.head = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(model.head.in_features, NUM_OF_CLASSES)
        )

    return model

## 11. Training Pipeline

The `fit_predict` function handles the complete training lifecycle for any model:

**Training strategy:**
- `BCEWithLogitsLoss` with `pos_weight` — penalizes missed rare diseases more heavily
- `AdamW` with `weight_decay=1e-2` — L2 regularization to reduce overfitting
- `CosineAnnealingLR` — gradually reduces learning rate from `LR` to `1e-6`
- Mixed precision (`autocast` + `GradScaler`) — doubles training speed on GPU
- Early stopping on validation AUC — saves the best weights automatically

**For Inception V3 specifically:**  
The model returns both main and auxiliary outputs during training.  
Total loss = `main_loss + 0.4 × auxiliary_loss` (standard Inception practice).

**Evaluation metric:** Mean AUC-ROC across all valid classes (excluding classes
with no positive samples in the validation set to avoid undefined AUC).

In [ ]:
def fit_predict(model_name, model_weights, all_splitted_data,
                show_logs=True, show_results=True):
    """
    Full training and evaluation pipeline for any supported architecture.

    Steps:
    1. Build and modify model
    2. Train with early stopping
    3. Load best weights
    4. Find optimal thresholds on validation set
    5. Evaluate on test set
    6. Save results to CSV and model to .pth

    Parameters
    ----------
    model_name       : torchvision model function (e.g. resnet50)
    model_weights    : corresponding weights enum (e.g. ResNet50_Weights)
    all_splitted_data: list — [train_dl, val_dl, test_dl]
    show_logs        : bool — print epoch-by-epoch progress
    show_results     : bool — plot curves, confusion matrix, and save results
    """
    model = model_name(model_weights.DEFAULT)
    name  = model.__class__.__name__
    model = replace_classifier(model, name)
    model = model.to(dev)

    # Use DataParallel if multiple GPUs are available
    if cuda.device_count() > 1:
        model = nn.DataParallel(model)

    # BCEWithLogitsLoss: suitable for multi-label — treats each class independently
    # pos_weight: upweights rare diseases to address class imbalance
    criterion = nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHTS, reduction='mean')

    # AdamW with weight decay for regularization
    opt = AdamW(model.parameters(), lr=LR, weight_decay=1e-2)

    # Cosine annealing: smoothly reduces LR from LR to eta_min over all epochs
    scheduler = lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-6)

    best_val_auc = 0.0
    counter      = 0

    save_best_model_path = os.path.join(SAVED_MODELS_FOLDER, f'best_{name}_model.pth')

    if show_logs:
        print(f"{name} Is Running")

    history = []
    scaler  = GradScaler()
    start   = time.time()

    for epoch in range(EPOCHS):

        # ── TRAINING ──────────────────────────────────────────────────────────
        all_train_loss   = []
        all_train_preds  = []
        all_train_labels = []
        model.train()

        for batch in tqdm(all_splitted_data[0],
                          desc=f"Epoch {epoch+1}/{EPOCHS}" if show_logs else None):
            imgs, labels = batch
            imgs   = imgs.to(dev, non_blocking=True)
            labels = labels.to(dev, non_blocking=True)

            opt.zero_grad()

            with autocast(device_type='cuda'):
                # Inception V3 returns (main_output, aux_output) during training
                outputs, aux_outputs = model(imgs)
                loss1 = criterion(outputs, labels)
                loss2 = criterion(aux_outputs, labels)
                # Auxiliary loss weighted by 0.4 — standard Inception training
                loss  = loss1 + 0.4 * loss2

            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

            all_train_loss.append(loss.item())
            all_train_preds.append(sigmoid(outputs).detach().cpu())
            all_train_labels.append(labels.detach().cpu())

        # Step the LR scheduler once per epoch
        scheduler.step()

        all_train_preds  = cat(all_train_preds).numpy()
        all_train_labels = cat(all_train_labels).numpy()

        # Compute AUC only on classes with at least one positive sample
        valid_classes = [i for i in range(NUM_OF_CLASSES)
                         if all_train_labels[:, i].sum() > 0]
        train_auc  = roc_auc_score(all_train_labels[:, valid_classes],
                                   all_train_preds[:, valid_classes], average="macro")
        train_loss = sum(all_train_loss) / len(all_train_loss)

        # ── VALIDATION ────────────────────────────────────────────────────────
        all_val_loss   = []
        all_val_preds  = []
        all_val_labels = []
        model.eval()

        with no_grad():
            for batch in all_splitted_data[1]:
                imgs, labels = batch
                imgs   = imgs.to(dev, non_blocking=True)
                labels = labels.to(dev, non_blocking=True)

                with autocast(device_type='cuda'):
                    outputs = model(imgs)
                    loss    = criterion(outputs, labels)

                all_val_loss.append(loss.item())
                all_val_preds.append(sigmoid(outputs).detach().cpu())
                all_val_labels.append(labels.detach().cpu())

        val_loss = sum(all_val_loss) / len(all_val_loss)
        all_val_preds  = cat(all_val_preds).numpy()
        all_val_labels = cat(all_val_labels).numpy()

        valid_classes = [i for i in range(NUM_OF_CLASSES)
                         if all_val_labels[:, i].sum() > 0]
        val_auc = roc_auc_score(all_val_labels[:, valid_classes],
                                all_val_preds[:, valid_classes], average="macro")

        if show_logs:
            print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
                  f"Train AUC: {train_auc:.4f} | Val AUC: {val_auc:.4f}")

        history.append({
            "train_loss": train_loss, "val_loss": val_loss,
            "train_auc":  train_auc,  "val_auc":  val_auc,
        })

        # ── EARLY STOPPING ────────────────────────────────────────────────────
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            counter = 0
            save(model.state_dict(), save_best_model_path)
            if show_logs:
                print(f" New best model saved! AUC: {best_val_auc:.4f}")
        else:
            counter += 1
            if show_logs:
                print(f"  No improvement ({counter}/{PATIENCE})")
            if counter >= PATIENCE:
                if show_logs:
                    print("Early Stopping!!")
                break

    # ── POST-TRAINING EVALUATION ──────────────────────────────────────────────
    end   = time.time()
    hours = (end - start) / 3600

    if show_results:
        # Load best weights saved during training
        model.load_state_dict(load(save_best_model_path, map_location=dev))
        print("Best Model Loaded")

        Show_Curves(history)

        # Get optimal thresholds from VALIDATION set
        all_val_labels, all_val_preds, best_thresholds = model_predict(model, all_splitted_data[1])

        # Run final evaluation on TEST set
        all_test_labels, all_test_preds, _ = model_predict(model, all_splitted_data[2])

        # Apply validation thresholds to test predictions
        all_preds_binary = np.zeros_like(all_test_preds, dtype=np.float32)
        for i in range(NUM_OF_CLASSES):
            all_preds_binary[:, i] = (all_test_preds[:, i] > best_thresholds[i]).astype(int)

        # Compute test metrics
        valid_classes = [i for i in range(NUM_OF_CLASSES)
                         if all_test_labels[:, i].sum() > 0]

        results_row = {
            'Model'    : name,
            'AUC'      : np.round(roc_auc_score(all_test_labels[:, valid_classes],
                                                 all_test_preds[:, valid_classes],
                                                 average="macro"), 4),
            'F1'       : np.round(f1_score(all_test_labels, all_preds_binary,
                                           average='macro', zero_division=0), 4),
            'Precision': np.round(precision_score(all_test_labels, all_preds_binary,
                                                   average='macro', zero_division=0), 4),
            'Recall'   : np.round(recall_score(all_test_labels, all_preds_binary,
                                               average='macro', zero_division=0), 4),
            'Training_Time (Hours)': np.round(hours, 2),
        }

        Show_MCM(all_test_labels, all_preds_binary)

        # Append results to the shared CSV file
        results_path = r'C:\Users\ibrah.HIMA\OneDrive\Desktop\Full AI\My Reserch Papers\chest-xray-multilabel-classification\results\All_Models_Results.csv'
        results_df = pd.read_csv(results_path) if os.path.exists(results_path) else pd.DataFrame()
        results_df = pd.concat([results_df, pd.DataFrame([results_row])], ignore_index=True)
        results_df.to_csv(results_path, index=False)
        print(results_df)

        # Save model weights + test results to a single checkpoint file
        try:
            save({
                'model_state': model.state_dict(),
                'test_preds' : all_preds_binary,   # binary (0/1) — for metrics
                'test_probs' : all_test_preds,      # float probabilities — for AUC & Grad-CAM
                'test_labels': all_test_labels,
            }, save_best_model_path)
        except:
            # If saving fails due to disk space, remove old file and retry
            os.remove(save_best_model_path)
            save({
                'model_state': model.state_dict(),
                'test_preds' : all_preds_binary,
                'test_probs' : all_test_preds,
                'test_labels': all_test_labels,
            }, save_best_model_path)

## 12. Run Training

Run each model separately. Results are automatically appended to `All_Models_Results.csv`.

**Order of training:**
1. ResNet-50 (baseline CNN)
2. VGG-19 (baseline CNN)
3. ViT-B/16 (baseline Transformer)
4. Inception V3 (baseline CNN with auxiliary classifier)
5. Swin Transformer V2-S (proposed model)

In [ ]:
# Uncomment the model you want to train and run this cell

# fit_predict(resnet50,    ResNet50_Weights,    [train_dl,      val_dl,      test_dl])
# fit_predict(vgg19,       VGG19_Weights,       [train_dl,      val_dl,      test_dl])
# fit_predict(vit_b_16,    ViT_B_16_Weights,    [train_dl,      val_dl,      test_dl])
# fit_predict(inception_v3, Inception_V3_Weights, [inc_train_dl, inc_val_dl, inc_test_dl])
# fit_predict(swin_v2_s,   Swin_V2_S_Weights,   [swin_train_dl, swin_val_dl, swin_test_dl])